In [ ]:
import pandas as pd 

movies = pd.read_csv("movies_final.csv")
movies.head()

,genres,id,original_language,description,poster_path,release_date,duration,title,rating
0,"Animation, Comedy, Family",862,en,"Led by Woody, Andy's toys live happily in his ...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,1995-10-30,81.0,Toy Story,7.7
1,"Adventure, Fantasy, Family",8844,en,When siblings Judy and Peter discover an encha...,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,1995-12-15,104.0,Jumanji,6.9
2,"Romance, Comedy",15602,en,A family wedding reignites the ancient feud be...,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,1995-12-22,101.0,Grumpier Old Men,6.5
3,"Comedy, Drama, Romance",31357,en,"Cheated on, mistreated and stepped on, the wom...",/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,1995-12-22,127.0,Waiting to Exhale,6.1
4,Comedy,11862,en,Just when George Banks has recovered from his ...,/e64sOI48hQXyru7naBFyssKFxVd.jpg,1995-02-10,106.0,Father of the Bride Part II,5.7


In [ ]:
%pip install sentence-transformers

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 2.6 MB/s eta 0:00:04
   ---- ----------------------------------- 1.0/10.4 MB 2.8 MB/s eta 0:00:04
   ------ --------------------------------- 1.6/10.4 MB 2.6 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/10.4 MB 2.5 MB/s eta 0:00:04
   ----------- ---------------------------- 2.9/10.4 MB 2.9 MB/s eta 0:00:03
   ---------------- ----------------------- 4.2/10.4 MB 3.5 MB/s eta 0:00:02
   ----------------------- ---------------- 6.0/10.4 MB 4.3 MB/s eta 0:00:02
   ------------------------------ --------- 7.9/10.4 MB 4.8 MB/s eta 0:00:01
   -------------------------------------- - 10.0/10.4 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 5.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/553.3 kB ? eta -:--:--
   --------------------------------------- 553.3/553.3 kB 40.1 MB/s eta 0:00:00
   ----

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as numpy

model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
movies['soup'] = (
    movies['genres'].fillna('') + '. ' +
    movies['description'].fillna('')
).str.lower().str.replace('\n', ' ', regex=False)
movies.iloc[0]


genres                                       Animation, Comedy, Family
id                                                                 862
original_language                                                   en
description          Led by Woody, Andy's toys live happily in his ...
poster_path                           /rhIRbceoE9lR4veEXuwCC2wARtG.jpg
release_date                                                1995-10-30
duration                                                          81.0
title                                                        Toy Story
rating                                                             7.7
soup                 animation, comedy, family. led by woody, andy'...
Name: 0, dtype: object

In [11]:
embeddings = model.encode(movies['soup'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/1379 [00:00<?, ?it/s]

In [21]:
%pip install faiss-cpu


   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.8/18.9 MB 3.2 MB/s eta 0:00:06
   -- ------------------------------------- 1.3/18.9 MB 2.3 MB/s eta 0:00:08
   --- ------------------------------------ 1.6/18.9 MB 2.1 MB/s eta 0:00:09
   ---- ----------------------------------- 2.4/18.9 MB 2.4 MB/s eta 0:00:07
   ------- -------------------------------- 3.7/18.9 MB 3.1 MB/s eta 0:00:05
   --------- ------------------------------ 4.7/18.9 MB 3.5 MB/s eta 0:00:05
   ------------- -------------------------- 6.3/18.9 MB 4.0 MB/s eta 0:00:04
   ----------------- ---------------------- 8.1/18.9 MB 4.5 MB/s eta 0:00:03
   --------------------- ------------------ 10.0/18.9 MB 5.0 MB/s eta 0:00:02
   ------------------------- -------------- 12.1/18.9 MB 5.5 MB/s eta 0:00:02
   ------------------------------ --------- 14.4/18.9 MB 6.0 MB/s eta 0:00:01
   ------

In [23]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype('float32')
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  
faiss.normalize_L2(embeddings)
index.add(embeddings)

In [27]:
def recommend_movie_faiss(title, top_k=10, sim_weight=0.7, rating_weight=0.3):

    idx = movies[movies['title'] == title].index[0]

    query_vector = embeddings[idx].reshape(1, -1)

    faiss.normalize_L2(query_vector)

    similarity_scores, indices = index.search(query_vector, top_k+1)

    indices = indices[0][1:]  
    similarity_scores = similarity_scores[0][1:]

    max_rating = movies['rating'].max()

    weighted = []

    for i, sim_score in zip(indices, similarity_scores):

        rating_score = movies.iloc[i]['rating'] / max_rating

        final_score = sim_weight * sim_score + rating_weight * rating_score

        weighted.append((i, final_score))

    weighted = sorted(weighted, key=lambda x: x[1], reverse=True)

    movie_indices = [i[0] for i in weighted]

    return movies.iloc[movie_indices]['title'].values.tolist()

recommend_movie_faiss("One Piece Film Strong World")

['Baahubali: The Beginning',
 'Hellboy II: The Golden Army',
 'Percy Jackson: Sea of Monsters',
 "Atlantis: Milo's Return",
 '10,000 BC',
 'Hammer of the Gods',
 'The Scorpion King: Rise of a Warrior',
 'The Loves of Hercules',
 'Sinbad of the Seven Seas',
 'Dragonquest']